# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset is provided in a Croissant schema. All references to record sets, fields, and columns use their Croissant `@id`s for consistency and reproducibility.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

# Standard data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset metadata and display name/description
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This step helps you understand the structure of the data before loading into DataFrames.

In [ ]:
# List the available record sets and fields by `@id`
print("Available Record Sets:")
for record_set in metadata.record_sets:
    print(f"  - Name: {record_set.name}\n    @id: {record_set.id}")
    print("    Fields:")
    for field in record_set.fields:
        print(f"      - {field.name} (field @id: {field.id}, data type: {field.data_type})")
    print()

You can also inspect a sample of the records available in a given record set using its `@id`. Example below shows how to preview records by `@id`.

In [ ]:
# Choose the main record set @id (usually demographic or survey data)
record_set_ids = [r.id for r in metadata.record_sets]

# For illustration, preview the first record set
record_set_id = record_set_ids[0]

print(f"First 2 records in record set @id: {record_set_id}")
for i, record in enumerate(dataset.records(record_set=record_set_id)):
    print(record)
    if i >= 1:
        break

## 3. Data Extraction
Extract data from selected record set(s) for analysis. All record set, field, and column references below use their `@id`s as shown above.

In [ ]:
# Load all main record sets into separate DataFrames keyed by their `@id`
dataframes = {}

for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Loaded DataFrame for record set @id: {recset_id}, columns: {df.columns.tolist()}")
    else:
        print(f"Record set @id: {recset_id} is empty.")

# Display first few rows of the main DataFrame
main_record_set = record_set_ids[0]
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps, such as filtering records, handling numeric columns, and grouping/categorizing by attributes. All columns are referenced by their `@id`.

In [ ]:
# --- Identify numeric and grouping fields by their `@id` ---
main_df = dataframes[main_record_set]

# List numeric columns and a possible group field by their @id from overview
numeric_cols = main_df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    # Auto-detect possible numeric columns even if loaded as strings
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col])
            numeric_cols.append(col)
        except Exception:
            continue

print("Available numeric fields (by @id):", numeric_cols)
numeric_field = numeric_cols[0] if numeric_cols else None

# Example filter: analyze only rows where the main mental health score > threshold
if numeric_field:
    threshold = 5
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize the column
    col_normed = f"{numeric_field}_normalized"
    filtered_df[col_normed] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, col_normed]].head())
    
    # Attempt to group by a categorical field (first non-numeric field)
    group_fields = [c for c in main_df.columns if c != numeric_field and main_df[c].nunique() < 10]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field and any relationships to a grouping attribute.

In [ ]:
if numeric_field:
    plt.figure(figsize=(8, 5))
    main_df[numeric_field].dropna().hist(bins=20)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 5))
        main_df.boxplot(column=numeric_field, by=group_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² Mental Health Survey Dataset using Croissant `@id` referencing throughout. You examined records, extracted survey tables into DataFrames, and visualized selected fields.

- Key record sets and fields (by `@id`) enable programmatic reproducibility with `mlcroissant`.
- You can further extend EDA and analysis by referencing additional fields, performing missing data handling, or modeling.

For details, see [Croissant documentation](https://mlcommons.org/croissant) or the [dataset source](https://sen.science/doi/10.71728/senscience.vcs2-05nj).